## 1. Environment Setup

In [1]:
import warnings
warnings.filterwarnings('ignore')
import os
import json
import time
import hashlib
import numpy as np
import pandas as pd
from pathlib  import Path
from typing   import Optional
from dotenv   import load_dotenv
from tqdm     import tqdm

from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct
)

In [2]:
load_dotenv()

QDRANT_URL     = os.getenv('QDRANT_URL')
QDRANT_API_KEY = os.getenv('QDRANT_API_KEY')

assert QDRANT_URL,     'QDRANT_URL missing from .env'
assert QDRANT_API_KEY, 'QDRANT_API_KEY missing from .env'

MODULE_4_DIR   = Path('../module_4_rag')
DATASET_PATH   = MODULE_4_DIR / 'clean_dataset.parquet'
META_PATH      = MODULE_4_DIR / 'dataset_meta.json'
CACHE_PATH     = MODULE_4_DIR / 'embedding_cache.json'

assert DATASET_PATH.exists(), f'Dataset not found: {DATASET_PATH}'
assert META_PATH.exists(),    f'Meta not found: {META_PATH}'

print(f'QDRANT_URL     : {QDRANT_URL}')
print(f'QDRANT_API_KEY : set')
print(f'Dataset path   : {DATASET_PATH}  (exists: {DATASET_PATH.exists()})')
print(f'Cache path     : {CACHE_PATH}')

QDRANT_URL     : https://c80aee59-6dde-4b36-b285-a6754c9982cd.eu-west-2-0.aws.cloud.qdrant.io/
QDRANT_API_KEY : set
Dataset path   : ..\module_4_rag\clean_dataset.parquet  (exists: True)
Cache path     : ..\module_4_rag\embedding_cache.json


## 2. Load Clean Dataset

Loading the parquet exported from the data prep notebook and verifying  
that the row count, columns and types are exactly as expected

In [3]:
df = pd.read_parquet(DATASET_PATH)

with open(META_PATH) as f:
    meta = json.load(f)

print(f'Documents loaded  : {len(df):,}')
print(f'Expected          : {meta["n_documents"]:,}')
print(f'Match             : {len(df) == meta["n_documents"]}')
print()
print('Columns:')
print(df.dtypes.to_string())
print()


print('Risk distribution:')
for level in ['crisis', 'high', 'medium', 'low']:
    count = (df['risk_level'] == level).sum()
    print(f'  {level:<8} : {count:>4}  ({count/len(df)*100:.1f}%)')

    
print()
print('Quality distribution:')
for score in range(4):
    count = (df['quality_score'] == score).sum()
    print(f'  Score {score}  : {count:>4}  ({count/len(df)*100:.1f}%)')

Documents loaded  : 938
Expected          : 938
Match             : True

Columns:
doc_id             int64
Context              str
Response             str
document_text        str
topics            object
risk_level           str
quality_score      int32
has_empathy         bool
has_actionable      bool
context_len        int64
response_len       int64
document_len       int64
context_words      int64
response_words     int64

Risk distribution:
  crisis   :   20  (2.1%)
  high     :   69  (7.4%)
  medium   :  314  (33.5%)
  low      :  535  (57.0%)

Quality distribution:
  Score 0  :    3  (0.3%)
  Score 1  :  271  (28.9%)
  Score 2  :  449  (47.9%)
  Score 3  :  215  (22.9%)


## 3. Embedding Generation

Loading "all-MiniLM-L6-v2" and embedding all 938 document_text values 

The EmbeddingCache class:
- Keys every embedding by the MD5 hash of its input text
- Saves to embedding_cache.json after all documents are processed
- On rerun serves all embeddings from cache instantly with zero model calls
- Also used at inference time to cache query embeddings during retrieval

In [4]:
EMBED_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
embed_model = SentenceTransformer(EMBED_MODEL_NAME)
VECTOR_DIM   = embed_model.get_sentence_embedding_dimension()

print(f'Model      : {EMBED_MODEL_NAME}')
print(f'Vector dim : {VECTOR_DIM}')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model      : sentence-transformers/all-MiniLM-L6-v2
Vector dim : 384


In [ ]:
class EmbeddingCache:
    def __init__(self, cache_file: Path):
        self.cache_file = cache_file
        self.cache      = self._load()
        self.hits       = 0
        self.misses     = 0


    def _hash(self, text: str) -> str:
        return hashlib.md5(text.encode('utf-8')).hexdigest()


    def get(self, text: str) -> Optional[list]:
        result = self.cache.get(self._hash(text))
        if result is not None:
            self.hits += 1
        else:
            self.misses += 1
        return result


    def set(self, text: str, embedding: list) -> None:
        self.cache[self._hash(text)] = embedding


    def embed(self, text: str) -> list:
        cached = self.get(text)
        if cached is not None:
            return cached
        embedding = embed_model.encode(text, normalize_embeddings = True).tolist()
        self.set(text, embedding)
        return embedding


    def save(self) -> None:
        with open(self.cache_file, 'w') as f:
            json.dump(self.cache, f)


    def _load(self) -> dict:
        try:
            with open(self.cache_file) as f:
                data = json.load(f)
                print(f'Cache loaded : {len(data):,} existing embeddings')
                return data
        except Exception:
            print('Cache is empty so all embeddings will be computed fresh')
            return {}


    @property
    def hit_rate(self) -> float:
        total = self.hits + self.misses
        return self.hits / total if total > 0 else 0.0

In [6]:
embedding_cache = EmbeddingCache(CACHE_PATH)
print(f'Cache file : {CACHE_PATH}')

Cache is empty — all embeddings will be computed fresh
Cache file : ..\module_4_rag\embedding_cache.json


In [7]:
# Embed all documents and Batched encoding is faster than one-by-one for fresh runs

texts      = df['document_text'].tolist()
embeddings = []
BATCH_SIZE = 64

all_cached = all(embedding_cache.get(t) is not None for t in texts)

if all_cached:
    print('All embeddings found in cache (loading instantly)')
    embeddings = [embedding_cache.get(t) for t in texts]
    print(f'Loaded {len(embeddings):,} embeddings from cache')


else:
    print(f'Computing embeddings for {len(texts):,} documents')
    t_start = time.time()

    for i in tqdm(range(0, len(texts), BATCH_SIZE), desc = 'Embedding'):
        batch       = texts[i : i + BATCH_SIZE]
        batch_vecs  = embed_model.encode(
            batch,
            normalize_embeddings = True,
            show_progress_bar    = False
        )
        for text, vec in zip(batch, batch_vecs):
            embedding_cache.set(text, vec.tolist())
            embeddings.append(vec.tolist())

    embedding_cache.save()
    elapsed = time.time() - t_start
    print(f'Done in {elapsed:.1f}s — cache saved to {CACHE_PATH}')


df['embedding'] = embeddings

print()
print(f'Total embeddings    : {len(embeddings):,}')
print(f'Vector dimension    : {len(embeddings[0])}')
print(f'Cache hit rate      : {embedding_cache.hit_rate:.2%}')
print(f'Cache size on disk  : {CACHE_PATH.stat().st_size / 1024 / 1024:.1f} MB' if CACHE_PATH.exists() else '')

Computing embeddings for 938 documents


Embedding: 100%|██████████| 15/15 [00:19<00:00,  1.27s/it]


Done in 19.5s — cache saved to ..\module_4_rag\embedding_cache.json

Total embeddings    : 938
Vector dimension    : 384
Cache hit rate      : 0.00%
Cache size on disk  : 7.6 MB


## 4. Qdrant Connection & Collection Setup

- Connecting to the free Qdrant cloud cluster and creating the collection.  
- The collection uses cosine distance which is standard for sentence-transformer embeddings.  
- If the collection already exists the creation step is skipped safely.

In [8]:
qdrant = QdrantClient(
    url     = QDRANT_URL,
    api_key = QDRANT_API_KEY,
    timeout = 60
)

COLLECTION_NAME = 'mental_health_counseling'
existing_collections = [c.name for c in qdrant.get_collections().collections]
print(f'Existing collections : {existing_collections}')

if COLLECTION_NAME not in existing_collections:
    qdrant.create_collection(
        collection_name = COLLECTION_NAME,
        vectors_config  = VectorParams( size = VECTOR_DIM , distance = Distance.COSINE)
    )
    print(f'Collection created : {COLLECTION_NAME}')

else:
    print(f'Collection exists  : {COLLECTION_NAME} so skipping creation')



info = qdrant.get_collection(COLLECTION_NAME)
print(f'Vectors in collection : {info.points_count:,}')
print(f'Vector size           : {info.config.params.vectors.size}')
print(f'Distance metric       : {info.config.params.vectors.distance}')

Existing collections : []
Collection created : mental_health_counseling
Vectors in collection : 0
Vector size           : 384
Distance metric       : Cosine


## 5. Indexing

- Uploading all 938 vectors to Qdrant in batches of 64.  
- Each point stores the dense vector plus the full metadata payload so the retriever has everything it needs at query time without a separate database lookup.

In [11]:
info = qdrant.get_collection(COLLECTION_NAME)

if info.points_count >= len(df):
    print(f'Collection already fully indexed ({info.points_count:,} vectors). Skipping.')

else:
    print(f'Indexing {len(df):,} documents into "{COLLECTION_NAME}"')
    BATCH_SIZE = 64
    t_start    = time.time()

    for i in tqdm(range(0, len(df), BATCH_SIZE), desc = 'Indexing'):
        batch  = df.iloc[i : i + BATCH_SIZE]
        points = []

        for _, row in batch.iterrows():
            vector = row['embedding']
            if hasattr(vector, 'tolist'):
                vector = vector.tolist()

            points.append(PointStruct(
                id      = int(row['doc_id']),
                vector  = vector,
                payload = {
                    'context'       : str(row['Context']),
                    'response'      : str(row['Response']),
                    'topics'        : list(row['topics']),
                    'risk_level'    : str(row['risk_level']),
                    'quality_score' : int(row['quality_score']),
                    'has_empathy'   : bool(row['has_empathy']),
                    'has_actionable': bool(row['has_actionable']),
                    'context_len'   : int(row['context_len']),
                    'response_len'  : int(row['response_len'])
                }
            ))

        qdrant.upsert(collection_name = COLLECTION_NAME, points = points)

    elapsed = time.time() - t_start
    print(f'Indexing complete in {elapsed:.1f}s')

final_info = qdrant.get_collection(COLLECTION_NAME)
print()
print(f'Total vectors indexed : {final_info.points_count:,}')
print(f'Expected              : {len(df):,}')
print(f'Match                 : {final_info.points_count == len(df)}')

Indexing 938 documents into "mental_health_counseling"...


Indexing: 100%|██████████| 15/15 [00:17<00:00,  1.16s/it]

Indexing complete in 17.4s

Total vectors indexed : 938
Expected              : 938
Match                 : True


## 6. Retrieval Smoke Test

- Running test queries against the live collection to verify that the index is working correctly and returning relevant documents.  
- Each result shows similarity score, risk level, topics and a preview  
of the context and response.

In [14]:
def retrieve(query: str, top_k: int = 3) -> list:
    query_vector = embedding_cache.embed(query)
    results      = qdrant.query_points(
        collection_name = COLLECTION_NAME,
        query           = query_vector,
        limit           = top_k,
        with_payload    = True
    ).points

    return [
        {
            'similarity'  : round(r.score, 4),
            'risk_level'  : r.payload['risk_level'],
            'topics'      : r.payload['topics'],
            'quality'     : r.payload['quality_score'],
            'context'     : r.payload['context'],
            'response'    : r.payload['response']
        }
        for r in results
    ]

In [15]:
smoke_queries = [
    ('anxiety / sleep',   'I have been feeling very anxious and cannot sleep at night'),
    ('depression',        'I feel completely empty and hopeless every single day'),
    ('grief',             'I lost my mother recently and I do not know how to cope'),
    ('crisis',            'I have been thinking about ending my life'),
    ('out of scope',      'What is the capital of France'),
]

print('Retrieval Smoke Test\n')
print('=' * 80)

for label, query in smoke_queries:
    results = retrieve(query, top_k = 3)
    print(f'Query [{label}]')
    print(f'  "{query}"')
    print()
    for i, r in enumerate(results):
        print(f'  Result {i+1}')
        print(f'    Similarity : {r["similarity"]}')
        print(f'    Risk level : {r["risk_level"]}')
        print(f'    Topics     : {r["topics"]}')
        print(f'    Quality    : {r["quality"]}')
        print(f'    Context    : {r["context"][:100]}...')
        print(f'    Response   : {r["response"][:120]}...')
        print()
    print('=' * 80)
    print()

Retrieval Smoke Test

Query [anxiety / sleep]
  "I have been feeling very anxious and cannot sleep at night"

  Result 1
    Similarity : 0.6455
    Risk level : medium
    Topics     : ['anxiety', 'sleep', 'anger', 'stress']
    Quality    : 3
    Context    : I have a lot of issues going on right now. First of all, I have a lot of trouble sleeping at times, ...
    Response   : It sounds like you are noticing yourself becoming overwhelmed with anxiety, feeling more irritable, and struggling to sl...

  Result 2
    Similarity : 0.5998
    Risk level : medium
    Topics     : ['anxiety', 'anger', 'self_esteem', 'loneliness']
    Quality    : 3
    Context    : When I'm in large crowds I get angry and I just can't deal with people. I don't really like other pe...
    Response   : What you're experiencing is anxiety, it's actually quite common. Good news - you're not alone in this experience! That b...

  Result 3
    Similarity : 0.5939
    Risk level : medium
    Topics     : ['anxiet

## 7. Advanced Retrieval Engine

- Adaptive top-k  : query complexity drives how many chunks to fetch
- Similarity gate : chunks below 0.35 cosine similarity are flagged before wasting a judge call
- Emotion-aware reranker : free rescore using metadata already stored in Qdrant payload



In [17]:
# Similarity gate threshold (below this the results are noise)
SIMILARITY_GATE = 0.35

# Emotion rerank weights
EMOTION_RERANK_WEIGHTS = {
    'sadness' : {'risk_high': 0.15, 'risk_medium': 0.08, 'has_empathy': 0.12, 'quality': 0.02},
    'fear'    : {'risk_high': 0.15, 'risk_medium': 0.08, 'has_empathy': 0.12, 'quality': 0.02},
    'anger'   : {'risk_high': 0.05, 'risk_medium': 0.03, 'has_empathy': 0.15, 'quality': 0.02},
    'love'    : {'risk_high': 0.00, 'risk_medium': 0.00, 'has_empathy': 0.10, 'quality': 0.04},
    'joy'     : {'risk_high': 0.00, 'risk_medium': 0.00, 'has_empathy': 0.05, 'quality': 0.06},
    'surprise': {'risk_high': 0.00, 'risk_medium': 0.00, 'has_empathy': 0.05, 'quality': 0.06},
}
DEFAULT_WEIGHTS = {'risk_high': 0.05, 'risk_medium': 0.02, 'has_empathy': 0.08, 'quality': 0.03}


In [18]:
def compute_adaptive_top_k(query: str) -> int:
    """
    Returns top-k based on query complexity.
    Short simple queries need fewer chunks.
    Long compound queries need more context.
    """
    words         = query.lower().split()
    word_count    = len(words)
    compound_words = {'and', 'or', 'also', 'why', 'how', 'what', 'when', 'difference', 'between'}
    is_compound   = any(w in compound_words for w in words)

    if word_count <= 6 and not is_compound:
        return 3
    elif word_count <= 12 or not is_compound:
        return 5
    else:
        return 7

In [19]:
def emotion_aware_rerank(chunks: list, detected_emotion: str, emotion_confidence: float) -> list:
    """
    Rescores retrieved chunks using Qdrant metadata + Module 2 emotion signal.
    No API call pure arithmetic on existing scores.
    """
    weights = EMOTION_RERANK_WEIGHTS.get(detected_emotion, DEFAULT_WEIGHTS)

    # Apply confidence scaling  (low confidence emotion signal gets less weight)
    scale = min(emotion_confidence, 1.0)

    for chunk in chunks:
        boost = 0.0

        if chunk['risk_level'] == 'crisis':
            boost += weights['risk_high'] * scale * 1.5   # crisis chunks always prioritized
        elif chunk['risk_level'] == 'high':
            boost += weights['risk_high'] * scale
        elif chunk['risk_level'] == 'medium':
            boost += weights['risk_medium'] * scale

        if chunk['has_empathy']:
            boost += weights['has_empathy'] * scale

        boost += chunk['quality'] * weights['quality']

        chunk['reranked_score'] = round(chunk['similarity'] + boost, 4)

    return sorted(chunks, key = lambda x: x['reranked_score'], reverse = True)

In [20]:
def advanced_retrieve(
    query             : str,
    detected_emotion  : str   = 'neutral',
    emotion_confidence: float = 0.5,
    top_k_override    : int   = None
) -> dict:
    """
    Full retrieval pipeline with adaptive top-k, similarity gate and emotion reranker.

    Returns:
        dict with keys: chunks, top_k, gate_passed, gate_score, reranked
    """
    top_k        = top_k_override if top_k_override else compute_adaptive_top_k(query)
    query_vector = embedding_cache.embed(query)

    results = qdrant.query_points(
        collection_name = COLLECTION_NAME,
        query           = query_vector,
        limit           = top_k,
        with_payload    = True
    ).points

    chunks = [
        {
            'similarity'    : round(r.score, 4),
            'reranked_score': round(r.score, 4),
            'risk_level'    : r.payload['risk_level'],
            'topics'        : r.payload['topics'],
            'quality'       : r.payload['quality_score'],
            'has_empathy'   : r.payload['has_empathy'],
            'has_actionable': r.payload['has_actionable'],
            'context'       : r.payload['context'],
            'response'      : r.payload['response']
        }
        for r in results
    ]

    top_score   = chunks[0]['similarity'] if chunks else 0.0
    gate_passed = top_score >= SIMILARITY_GATE

    # Emotion-aware rerank
    chunks = emotion_aware_rerank(chunks, detected_emotion, emotion_confidence)

    return {
        'chunks'     : chunks,
        'top_k'      : top_k,
        'gate_passed': gate_passed,
        'gate_score' : top_score,
        'reranked'   : True
    }

In [21]:
# Verify advanced retrieval on the same smoke queries + emotion signal
print('Advanced Retrieval Engine Test')
print('=' * 80)

retrieval_tests = [
    ('anxiety + sadness',  'I have been feeling very anxious and cannot sleep',   'sadness', 0.94),
    ('grief + fear',       'I lost my mother and I am terrified of being alone',  'fear',    0.87),
    ('crisis high risk',   'I have been thinking about ending my life',            'sadness', 0.99),
    ('general + curious',  'What is cognitive behavioral therapy',                 'surprise',0.65),
    ('below gate',         'France capital Paris weather today',                   'joy',     0.55),
]

for label, query, emotion, conf in retrieval_tests:
    result = advanced_retrieve(query, detected_emotion = emotion, emotion_confidence = conf)
    print(f'Query [{label}]')
    print(f'  Text       : "{query}"')
    print(f'  Emotion    : {emotion} ({conf})')
    print(f'  Top-k      : {result["top_k"]}')
    print(f'  Gate score : {result["gate_score"]}  (passed: {result["gate_passed"]})')
    print()
    for i, c in enumerate(result['chunks'][:2]):
        print(f'  Chunk {i+1}')
        print(f'    Original score  : {c["similarity"]}')
        print(f'    Reranked score  : {c["reranked_score"]}')
        print(f'    Risk            : {c["risk_level"]}')
        print(f'    Has empathy     : {c["has_empathy"]}')
        print(f'    Topics          : {c["topics"]}')
    print('=' * 80)
    print()

Advanced Retrieval Engine Test
Query [anxiety + sadness]
  Text       : "I have been feeling very anxious and cannot sleep"
  Emotion    : sadness (0.94)
  Top-k      : 5
  Gate score : 0.6037  (passed: True)

  Chunk 1
    Original score  : 0.6037
    Reranked score  : 0.8517
    Risk            : medium
    Has empathy     : True
    Topics          : ['anxiety', 'sleep', 'anger', 'stress']
  Chunk 2
    Original score  : 0.5397
    Reranked score  : 0.7877
    Risk            : medium
    Has empathy     : True
    Topics          : ['anxiety', 'anger', 'self_esteem', 'loneliness']

Query [grief + fear]
  Text       : "I lost my mother and I am terrified of being alone"
  Emotion    : fear (0.87)
  Top-k      : 5
  Gate score : 0.5895  (passed: True)

  Chunk 1
    Original score  : 0.5195
    Reranked score  : 0.6639
    Risk            : low
    Has empathy     : True
    Topics          : ['anxiety']
  Chunk 2
    Original score  : 0.5895
    Reranked score  : 0.6295
    Risk    

## 8. Combined Intelligence Call

Single Groq call that acts as judge + query rewriter + quality scorer simultaneously.

Returns a structured JSON with an action field that drives the rest of the pipeline:
  - answer  → chunks are relevant, proceed to therapist LLM
  - rewrite → chunks are poor, use the rewritten_query field and search again
  - broaden → chunks are borderline, increase top-k by 2 and search again
  - fallback → chunks are completely irrelevant, answer from LLM general knowledge
  - crisis  → crisis signal detected, skip everything and respond with crisis support


In [22]:
from groq import Groq

GROQ_API_KEY  = os.getenv('GROQ_API_KEY')
assert GROQ_API_KEY, 'GROQ_API_KEY missing from .env'

groq_client   = Groq(api_key = GROQ_API_KEY)
JUDGE_MODEL   = 'llama-3.3-70b-versatile'
ANSWER_MODEL  = 'llama-3.3-70b-versatile'

JUDGE_SYSTEM_PROMPT = """You are a retrieval quality judge for a mental health support chatbot.

Your job is to evaluate whether the retrieved counseling passages are relevant and sufficient
to answer the user's query, and to decide what action the pipeline should take next.

You receive:
- user_query: the user's message
- detected_emotion: emotion detected by the classifier (sadness, fear, joy, anger, love, surprise)
- chunks: the top retrieved passages from the knowledge base

You must return ONLY a valid JSON object with this exact structure:
{
  "action": "answer | rewrite | broaden | fallback | crisis",
  "confidence": 0.0 to 1.0,
  "rewritten_query": "improved query string (only when action is rewrite, else null)",
  "reasoning": "one sentence explanation of your decision",
  "relevant_chunks": [0, 1, 2]  // indices of chunks that are actually useful (empty list if none)
}

Action decision rules:
- crisis   : user message contains suicidal ideation, self-harm intent, or immediate danger signals
- answer   : at least one chunk directly addresses the user's concern (confidence > 0.6)
- rewrite  : chunks are topically related but not quite right — a better query would find better results
- broaden  : chunks are partially relevant but thin — more results would help
- fallback : chunks are completely off-topic — answer from general mental health knowledge instead

Be conservative with crisis — only use it for clear and immediate danger, not general sadness."""

In [23]:
def combined_intelligence_call(
    query            : str,
    chunks           : list,
    detected_emotion : str,
    gate_passed      : bool
) -> dict:
    """
    Single Groq call: judge + rewriter + quality scorer.

    If gate did not pass (top similarity below 0.35) skip the API call
    and return rewrite action immediately to save latency and rate limits.
    """

    #  gate failed so no need to call the LLM
    if not gate_passed:
        return {
            'action'          : 'rewrite',
            'confidence'      : 0.0,
            'rewritten_query' : None,
            'reasoning'       : f'Similarity gate failed (top score below {SIMILARITY_GATE}) rewriting query',
            'relevant_chunks' : [],
            'gate_triggered'  : True
        }




    # Build chunk summary for the prompt
    chunk_summary = []
    for i, c in enumerate(chunks[:5]):   # max 5 chunks to keep prompt short
        chunk_summary.append(
            f"Chunk {i} [risk={c['risk_level']}, score={c['reranked_score']}]:\n"
            f"Context: {c['context'][:200]}\n"
            f"Response: {c['response'][:200]}"
        )

    user_message = json.dumps({
        'user_query'      : query,
        'detected_emotion': detected_emotion,
        'chunks'          : chunk_summary
    }, ensure_ascii = False)

    try:
        response = groq_client.chat.completions.create(
            model           = JUDGE_MODEL,
            messages        = [
                {'role': 'system', 'content': JUDGE_SYSTEM_PROMPT},
                {'role': 'user',   'content': user_message}
            ],
            temperature     = 0.0,
            max_tokens      = 300,
            response_format = {'type': 'json_object'}
        )


        result = json.loads(response.choices[0].message.content.strip())
        result['gate_triggered'] = False
        return result



    except Exception as e:
        # Safe fallback on any error
        return {
            'action'          : 'fallback',
            'confidence'      : 0.0,
            'rewritten_query' : None,
            'reasoning'       : f'Judge call failed: {str(e)[:80]}',
            'relevant_chunks' : [],
            'gate_triggered'  : False
        }

In [24]:
# Verify the combined intelligence call
print('Combined Intelligence Call Test')
print('=' * 80)

judge_tests = [
    ('clear anxiety question',  'I have been feeling very anxious and cannot sleep at night',   'sadness',  0.94),
    ('crisis signal',           'I want to end my life I cannot take this anymore',             'sadness',  0.99),
    ('vague query',             'things are bad',                                               'sadness',  0.71),
    ('completely off topic',    'What is the capital of France',                                'surprise', 0.55),
]

for label, query, emotion, conf in judge_tests:
    retrieval = advanced_retrieve(query, detected_emotion = emotion, emotion_confidence = conf)
    judgment  = combined_intelligence_call(
        query            = query,
        chunks           = retrieval['chunks'],
        detected_emotion = emotion,
        gate_passed      = retrieval['gate_passed']
    )

    
    print(f'Test [{label}]')
    print(f'  Query    : "{query}"')
    print(f'  Gate     : {retrieval["gate_score"]} (passed: {retrieval["gate_passed"]})')
    print(f'  Action   : {judgment["action"]}')
    print(f'  Confidence : {judgment["confidence"]}')
    print(f'  Reasoning  : {judgment["reasoning"]}')
    if judgment.get('rewritten_query'):
        print(f'  Rewritten  : {judgment["rewritten_query"]}')
    print('=' * 80)
    print()

Combined Intelligence Call Test
Test [clear anxiety question]
  Query    : "I have been feeling very anxious and cannot sleep at night"
  Gate     : 0.6455 (passed: True)
  Action   : answer
  Confidence : 0.8
  Reasoning  : The retrieved chunks directly address the user's concern about anxiety and sleep issues, providing relevant and sufficient information to answer the query.

Test [crisis signal]
  Query    : "I want to end my life I cannot take this anymore"
  Gate     : 0.3746 (passed: True)
  Action   : crisis
  Confidence : 1.0
  Reasoning  : The user's message explicitly expresses suicidal ideation, indicating a clear and immediate danger.

Test [vague query]
  Query    : "things are bad"
  Gate     : 0.2628 (passed: False)
  Action   : rewrite
  Confidence : 0.0
  Reasoning  : Similarity gate failed (top score below 0.35) rewriting query

Test [completely off topic]
  Query    : "What is the capital of France"
  Gate     : 0.1146 (passed: False)
  Action   : rewrite
  Confiden

## 9. Therapist LLM

The answer generation call with the full therapist system prompt.

This is where the warm therapist personality lives.

Every response is emotion-aware, language-aware, and crisis-aware.

In [28]:
# Crisis Hotlines  

TEST_NUMBER = '01011216969'   

CRISIS_HOTLINES = {

    # Arabic-speaking countries
    'EG' : {'name': 'Egypt Mental Health Hotline',         'number': TEST_NUMBER, 'language': 'ar'},
    'SA' : {'name': 'Saudi Mental Health Hotline',         'number': TEST_NUMBER, 'language': 'ar'},
    'AE' : {'name': 'UAE Mental Wellness Hotline',         'number': TEST_NUMBER, 'language': 'ar'},
    'JO' : {'name': 'Jordan LifeLine',                     'number': TEST_NUMBER, 'language': 'ar'},
    'MA' : {'name': 'Morocco Crisis Line',                 'number': TEST_NUMBER, 'language': 'ar'},
    'IQ' : {'name': 'Iraq Mental Health Line',             'number': TEST_NUMBER, 'language': 'ar'},
    'SY' : {'name': 'Syria Support Line',                  'number': TEST_NUMBER, 'language': 'ar'},
    'LB' : {'name': 'Lebanon Embrace Lifeline',            'number': TEST_NUMBER, 'language': 'ar'},
    'KW' : {'name': 'Kuwait Mental Health Line',           'number': TEST_NUMBER, 'language': 'ar'},
    'QA' : {'name': 'Qatar Mental Health Line',            'number': TEST_NUMBER, 'language': 'ar'},

    # English-speaking countries
    'US' : {'name': '988 Suicide and Crisis Lifeline',     'number': TEST_NUMBER, 'language': 'en'},
    'GB' : {'name': 'Samaritans',                          'number': TEST_NUMBER, 'language': 'en'},
    'AU' : {'name': 'Lifeline Australia',                  'number': TEST_NUMBER, 'language': 'en'},
    'CA' : {'name': 'Crisis Services Canada',              'number': TEST_NUMBER, 'language': 'en'},
    'NZ' : {'name': 'Lifeline New Zealand',                'number': TEST_NUMBER, 'language': 'en'},
    'IE' : {'name': 'Samaritans Ireland',                  'number': TEST_NUMBER, 'language': 'en'},

    # European countries
    'DE' : {'name': 'Telefonseelsorge Germany',            'number': TEST_NUMBER, 'language': 'de'},
    'FR' : {'name': 'Numero National Prevention Suicide',  'number': TEST_NUMBER, 'language': 'fr'},
    'ES' : {'name': 'Telefono de la Esperanza',            'number': TEST_NUMBER, 'language': 'es'},
    'IT' : {'name': 'Telefono Amico',                      'number': TEST_NUMBER, 'language': 'it'},
    'PT' : {'name': 'SOS Voz Amiga',                       'number': TEST_NUMBER, 'language': 'pt'},
    'NL' : {'name': 'Stichting 113 Zelfmoordpreventie',    'number': TEST_NUMBER, 'language': 'nl'},
    'BE' : {'name': 'Centre de Prevention du Suicide',     'number': TEST_NUMBER, 'language': 'fr'},
    'SE' : {'name': 'Mind Självmordslinjen',                'number': TEST_NUMBER, 'language': 'sv'},
    'NO' : {'name': 'Mental Helse Hjelpelinjen',            'number': TEST_NUMBER, 'language': 'no'},
    'DK' : {'name': 'Livslinien Denmark',                  'number': TEST_NUMBER, 'language': 'da'},
    'FI' : {'name': 'Mieli Mental Health Finland',         'number': TEST_NUMBER, 'language': 'fi'},
    'PL' : {'name': 'Telefon Zaufania Poland',             'number': TEST_NUMBER, 'language': 'pl'},
    'RU' : {'name': 'Russian Psychological Help Line',     'number': TEST_NUMBER, 'language': 'ru'},
    'UA' : {'name': 'Ukraine Mental Health Line',          'number': TEST_NUMBER, 'language': 'uk'},
    'GR' : {'name': 'Klimaka Greece',                      'number': TEST_NUMBER, 'language': 'el'},
    'TR' : {'name': 'Turkey Mental Health Line',           'number': TEST_NUMBER, 'language': 'tr'},

    # Asian countries
    'IN' : {'name': 'iCall India',                         'number': TEST_NUMBER, 'language': 'hi'},
    'JP' : {'name': 'Befrienders Japan',                   'number': TEST_NUMBER, 'language': 'ja'},
    'CN' : {'name': 'Beijing Suicide Research Center',     'number': TEST_NUMBER, 'language': 'zh'},
    'KR' : {'name': 'Korea Suicide Prevention Hotline',    'number': TEST_NUMBER, 'language': 'ko'},
    'PH' : {'name': 'Hopeline Philippines',                'number': TEST_NUMBER, 'language': 'tl'},
    'TH' : {'name': 'Thailand Mental Health Hotline',      'number': TEST_NUMBER, 'language': 'th'},
    'ID' : {'name': 'Into The Light Indonesia',            'number': TEST_NUMBER, 'language': 'id'},
    'MY' : {'name': 'Befrienders Kuala Lumpur',            'number': TEST_NUMBER, 'language': 'ms'},
    'PK' : {'name': 'Umang Pakistan Helpline',             'number': TEST_NUMBER, 'language': 'ur'},
    'BD' : {'name': 'Kaan Pete Roi Bangladesh',            'number': TEST_NUMBER, 'language': 'bn'},

    # African countries
    'ZA' : {'name': 'SADAG South Africa',                  'number': TEST_NUMBER, 'language': 'en'},
    'NG' : {'name': 'Mentally Aware Nigeria Initiative',   'number': TEST_NUMBER, 'language': 'en'},
    'KE' : {'name': 'Befrienders Kenya',                   'number': TEST_NUMBER, 'language': 'en'},
    'GH' : {'name': 'Mental Health Authority Ghana',       'number': TEST_NUMBER, 'language': 'en'},

    # Latin American countries
    'BR' : {'name': 'CVV Brazil',                          'number': TEST_NUMBER, 'language': 'pt'},
    'MX' : {'name': 'SAPTEL Mexico',                       'number': TEST_NUMBER, 'language': 'es'},
    'AR' : {'name': 'Centro de Asistencia al Suicida',     'number': TEST_NUMBER, 'language': 'es'},
    'CO' : {'name': 'Linea 106 Colombia',                  'number': TEST_NUMBER, 'language': 'es'},
    'CL' : {'name': 'Telefono de la Esperanza Chile',      'number': TEST_NUMBER, 'language': 'es'},

    # Default fallback for any unlisted country
    'DEFAULT' : {
        'name'     : 'International Crisis Support',
        'number'   : TEST_NUMBER,
        'language' : 'en'
    }
}


def get_crisis_number(country_code: str) -> dict:
    """
    Returns the crisis hotline info for a given country code.
    Falls back to DEFAULT if country not in the dictionary.
    """
    return CRISIS_HOTLINES.get(country_code.upper(), CRISIS_HOTLINES['DEFAULT'])


def format_crisis_message(country_code: str, language_code: str, user_name: str = None) -> str:
    """
    Builds the crisis number message to inject into the LLM response.
    Language-aware — message text matches the detected language.
    """
    hotline    = get_crisis_number(country_code)
    name_part  = f"{user_name}, " if user_name else ""

    templates = {
        'ar' : f"{name_part}أنت لست وحدك. يرجى التواصل مع خط دعم الصحة النفسية: {hotline['name']} على الرقم {hotline['number']}",
        'en' : f"{name_part}you are not alone. Please reach out to {hotline['name']} at {hotline['number']}",
        'fr' : f"{name_part}vous n'etes pas seul. Contactez {hotline['name']} au {hotline['number']}",
        'de' : f"{name_part}Sie sind nicht allein. Bitte wenden Sie sich an {hotline['name']}: {hotline['number']}",
        'es' : f"{name_part}no estas solo. Por favor contacta {hotline['name']} al {hotline['number']}",
        'it' : f"{name_part}non sei solo. Contatta {hotline['name']} al {hotline['number']}",
        'ru' : f"{name_part}вы не одни. Позвоните на {hotline['name']}: {hotline['number']}",
        'tr' : f"{name_part}yalniz degilsiniz. Lutfen {hotline['name']} hattini arayin: {hotline['number']}",
    }

    return templates.get(language_code, templates['en'])

In [29]:
# Verify
print("Crisis hotline lookup test:")
for country in ['EG', 'US', 'DE', 'IN', 'BR', 'XX']:
    info = get_crisis_number(country)
    print(f"  {country} : {info['name']} → {info['number']}")

print()
print("Crisis message formatting test:")
print(f"  AR : {format_crisis_message('EG', 'ar', 'Ahmed')}")
print(f"  EN : {format_crisis_message('US', 'en', 'Sarah')}")
print(f"  FR : {format_crisis_message('FR', 'fr', 'Marie')}")

Crisis hotline lookup test:
  EG : Egypt Mental Health Hotline → 01011216969
  US : 988 Suicide and Crisis Lifeline → 01011216969
  DE : Telefonseelsorge Germany → 01011216969
  IN : iCall India → 01011216969
  BR : CVV Brazil → 01011216969
  XX : International Crisis Support → 01011216969

Crisis message formatting test:
  AR : Ahmed, أنت لست وحدك. يرجى التواصل مع خط دعم الصحة النفسية: Egypt Mental Health Hotline على الرقم 01011216969
  EN : Sarah, you are not alone. Please reach out to 988 Suicide and Crisis Lifeline at 01011216969
  FR : Marie, vous n'etes pas seul. Contactez Numero National Prevention Suicide au 01011216969


In [30]:
THERAPIST_SYSTEM_PROMPT = """You are a warm, empathetic mental health support companion.

Your role is to provide compassionate, evidence-based support to people experiencing
emotional difficulties. You are NOT a replacement for professional therapy, but you
are a knowledgeable and caring presence that helps people feel heard and supported.

PERSONALITY:
- Warm and human — never clinical, never robotic, never generic
- Validating first — always acknowledge feelings before offering advice
- Hopeful — maintain a sense of hope without minimizing pain
- Honest — if you don't know something, say so with warmth
- Boundary-aware — gently remind users to seek professional help when appropriate

RESPONSE GUIDELINES:
- Always begin by acknowledging the user's emotional state
- Use the user's name naturally if provided (not every sentence, just warmly)
- Draw primarily from the retrieved counseling passages when provided
- If no retrieved context is provided, draw from general mental health knowledge
- Keep responses concise but complete — 3 to 5 paragraphs maximum
- End with one concrete, actionable suggestion when appropriate
- Never diagnose conditions
- Never recommend specific medications

LANGUAGE:
- Respond in exactly the same language the user wrote in
- If the user wrote in Arabic, respond entirely in Arabic
- Match the formality level of the user's message

CRISIS PROTOCOL (when crisis_active is True in context):
- Lead with immediate validation and presence
- Include the crisis number provided in the context naturally within the response
- Use safe messaging guidelines: offer hope, do not describe methods, provide resources
- Do not close with generic advice — close with direct connection to support

CITATION:
- When using retrieved passages, indicate naturally (e.g. "research in counseling suggests...")
- Do not cite chunk numbers or indices — weave the information in naturally"""


def build_context_block(
    chunks          : list,
    relevant_indices: list,
    judgment_action : str
) -> str:
    """
    Builds the retrieved context string to inject into the therapist prompt.
    Only includes chunks the judge marked as relevant.
    """
    if judgment_action == 'fallback' or not chunks:
        return 'No specific passages retrieved so draw from general mental health knowledge.'

    selected = [chunks[i] for i in relevant_indices if i < len(chunks)] or chunks[:3]

    lines = ['RETRIEVED COUNSELING PASSAGES:']
    for i, c in enumerate(selected, 1):
        lines.append(
            f'\nPassage {i} [topics: {", ".join(c["topics"])}]:\n'
            f'Patient context: {c["context"][:300]}\n'
            f'Counselor response: {c["response"][:400]}'
        )
    return '\n'.join(lines)

In [31]:
def therapist_llm(
    query            : str,
    chunks           : list,
    relevant_indices : list,
    judgment_action  : str,
    detected_emotion : str,
    emotion_conf     : float,
    response_tone    : str,
    language_code    : str,
    user_name        : str   = None,
    country_code     : str   = None,
    crisis_active    : bool  = False,
    prior_crisis     : bool  = False,
    conversation_history: list = None
) -> dict:
    """
    Generates the final therapist response.

    Args:
        query             : Current user message
        chunks            : Retrieved chunks from Qdrant
        relevant_indices  : Which chunks the judge approved
        judgment_action   : answer / fallback / crisis
        detected_emotion  : From Module 2
        emotion_conf      : From Module 2
        response_tone     : From Module 2 emotion_meta
        language_code     : From Module 1
        user_name         : From user account
        country_code      : From user account (for crisis number)
        crisis_active     : Current turn has crisis signal
        prior_crisis      : Any previous turn in session had crisis signal
        conversation_history : Last N turns for context
    """

    context_block = build_context_block(chunks, relevant_indices, judgment_action)

    # Build crisis injection
    crisis_block = ''
    if crisis_active or prior_crisis:
        hotline      = get_crisis_number(country_code or 'DEFAULT')
        crisis_block = (
            f'\nCRISIS CONTEXT ACTIVE:\n'
            f'Crisis number for user: {hotline["name"]} — {hotline["number"]}\n'
            f'Include this number naturally in your response.\n'
            f'Prior crisis in session: {prior_crisis}\n'
        )


    # Build conversation history block
    history_block = ''
    if conversation_history:
        lines = ['\nCONVERSATION HISTORY (most recent last):']
        for turn in conversation_history[-6:]:
            role    = turn.get('role', 'user')
            content = turn.get('content', '')[:200]
            lines.append(f'{role.upper()}: {content}')
        history_block = '\n'.join(lines)

    user_message = (
        f'USER NAME: {user_name or "not provided"}\n'
        f'DETECTED EMOTION: {detected_emotion} (confidence: {emotion_conf:.2f})\n'
        f'RESPONSE TONE: {response_tone}\n'
        f'LANGUAGE: {language_code}\n'
        f'{crisis_block}'
        f'{history_block}\n'
        f'\n{context_block}\n'
        f'\nUSER MESSAGE: {query}'
    )



    try:
        response = groq_client.chat.completions.create(
            model       = ANSWER_MODEL,
            messages    = [
                {'role': 'system', 'content': THERAPIST_SYSTEM_PROMPT},
                {'role': 'user',   'content': user_message}
            ],
            temperature = 0.4,   # slight creativity for natural warmth
            max_tokens  = 600
        )

        answer = response.choices[0].message.content.strip()

        # Build citations
        citations = []
        if judgment_action != 'fallback' and chunks:
            selected = [chunks[i] for i in relevant_indices if i < len(chunks)] or chunks[:3]
            for c in selected:
                citations.append({
                    'topics'    : c['topics'],
                    'risk_level': c['risk_level'],
                    'score'     : c['reranked_score']
                })

        return {
            'answer'   : answer,
            'citations': citations,
            'action'   : judgment_action,
            'language' : language_code
        }



    except Exception as e:
        # Graceful degradation so never return an empty response
        fallback_msg = (
            "I hear you, and I want you to know you are not alone. "
            "I'm having a technical difficulty right now but please don't hesitate "
            "to reach out again. Your wellbeing matters."
        )
        if crisis_active and country_code:
            hotline      = get_crisis_number(country_code)
            fallback_msg += f' If you need immediate support, please contact {hotline["name"]} at {hotline["number"]}.'

        return {
            'answer'   : fallback_msg,
            'citations': [],
            'action'   : 'fallback_error',
            'language' : language_code,
            'error'    : str(e)[:100]
        }

In [32]:
# Verify therapist LLM on two representative cases
print('Therapist LLM Test')
print('=' * 80)

therapist_tests = [
    {
        'label'   : 'anxiety with context',
        'query'   : 'I have been feeling very anxious and cannot sleep at night',
        'emotion' : 'fear', 'conf': 0.91, 'tone': 'Reassuring, grounding, structured',
        'lang'    : 'en', 'name': 'Sara', 'country': 'US', 'crisis': False
    },
    {
        'label'   : 'crisis case',
        'query'   : 'I feel like there is no point anymore and I want it all to stop',
        'emotion' : 'sadness', 'conf': 0.98, 'tone': 'Warm, validating, gentle',
        'lang'    : 'en', 'name': 'Ahmed', 'country': 'EG', 'crisis': True
    },
]

for t in therapist_tests:
    retrieval = advanced_retrieve(t['query'], t['emotion'], t['conf'])
    judgment  = combined_intelligence_call(t['query'], retrieval['chunks'], t['emotion'], retrieval['gate_passed'])
    result    = therapist_llm(
        query             = t['query'],
        chunks            = retrieval['chunks'],
        relevant_indices  = judgment.get('relevant_chunks', [0, 1, 2]),
        judgment_action   = judgment['action'],
        detected_emotion  = t['emotion'],
        emotion_conf      = t['conf'],
        response_tone     = t['tone'],
        language_code     = t['lang'],
        user_name         = t['name'],
        country_code      = t['country'],
        crisis_active     = t['crisis'],
        prior_crisis      = False
    )
    print(f'Test [{t["label"]}]')
    print(f'  Action     : {result["action"]}')
    print(f'  Citations  : {len(result["citations"])} chunks used')
    print()
    print(f'  RESPONSE:')
    print()
    for line in result['answer'].split('\n'):
        print(f'  {line}')
    print()
    print('=' * 80)
    print()

Therapist LLM Test
Test [anxiety with context]
  Action     : answer
  Citations  : 3 chunks used

  RESPONSE:

  Sara, I can sense the fear and anxiety that's been weighing on you, and I want you to know that I'm here to listen and support you. It takes a lot of courage to acknowledge and share your struggles, so thank you for reaching out. Research in counseling suggests that anxiety can be overwhelming, making it difficult to sleep and causing a sense of doom, as if it's hard to breathe. I'm here to reassure you that you're not alone in this experience.
  
  It's understandable that you're feeling this way, and it's great that you're recognizing the impact it's having on your daily life. Anxiety can be a sign of a current problem or a deeper emotional pattern, and it's possible that your mind is racing at night, making it hard to relax and fall asleep. I want to acknowledge that it's okay to feel this way, and it doesn't mean that there's anything wrong with you. In fact, many peopl

## 10. Session Memory

Sliding window  : last 6 turns kept in full  handles follow-ups and topic continuity
Sticky crisis   : once crisis fires it stays True for the entire session
Name extraction : detects user name from natural language if not in account


In [33]:
import re
import uuid


class SessionMemory:
    """
    Manages conversation state for a single user session
    """

    WINDOW_SIZE = 6

    def __init__(self, session_id: str = None, user_name: str = None, country_code: str = None):
        self.session_id       = session_id or str(uuid.uuid4())[:8]
        self.user_name        = user_name
        self.country_code     = country_code or 'DEFAULT'
        self.turns            = []
        self.crisis_flag      = False       
        self.emotion_history  = []
        self.topics_discussed = []
        self.session_topic    = None
        self.turn_count       = 0




    def add_turn(self, role: str, content: str, metadata: dict = None):
        """Add a turn to the sliding window"""
        self.turns.append({
            'role'    : role,
            'content' : content,
            'metadata': metadata or {}
        })
        # Keep only the last WINDOW_SIZE turns
        if len(self.turns) > self.WINDOW_SIZE:
            self.turns = self.turns[-self.WINDOW_SIZE:]
        self.turn_count += 1




    def update_from_pipeline(self, emotion: str, crisis_detected: bool, topics: list):
        """Update session state after each pipeline run"""

        self.emotion_history.append(emotion)

        if crisis_detected:
            self.crisis_flag = True

       
        for topic in topics:
            if topic not in self.topics_discussed:
                self.topics_discussed.append(topic)


        # Set session topic from first substantive turn
        if self.session_topic is None and len(self.topics_discussed) > 0:
            self.session_topic = self.topics_discussed[0]




    def extract_name_from_text(self, text: str) -> str | None:
        """Detect user name from natural language if not already known"""
        if self.user_name:
            return None   # already known

        patterns = [
            r"my name is (\w+)",
            r"i['']m (\w+)",
            r"i am (\w+)",
            r"call me (\w+)",
            r"this is (\w+)",
            r"اسمي\s+(\w+)",    # Arabic: my name is
            r"أنا\s+(\w+)",     # Arabic: I am
        ]

        for pattern in patterns:
            match = re.search(pattern, text.lower())

            if match:
                candidate = match.group(1).capitalize()
                # Filter out common false positives
                if candidate.lower() not in {'fine', 'okay', 'good', 'here', 'back', 'not', 'just'}:
                    self.user_name = candidate
                    return candidate
        return None



    def get_history_for_prompt(self) -> list:
        """Returns turns formatted for the therapist prompt"""
        return [
            {'role': t['role'], 'content': t['content']}
            for t in self.turns
        ]



    def dominant_emotion(self) -> str:
        """Returns the most frequent emotion in this session"""
        if not self.emotion_history:
            return 'neutral'
        from collections import Counter
        return Counter(self.emotion_history).most_common(1)[0][0]




    def summary(self) -> dict:
        """Returns session state summary for logging and email service"""
        return {
            'session_id'      : self.session_id,
            'user_name'       : self.user_name,
            'country_code'    : self.country_code,
            'turn_count'      : self.turn_count,
            'crisis_flag'     : self.crisis_flag,
            'dominant_emotion': self.dominant_emotion(),
            'emotion_history' : self.emotion_history,
            'topics_discussed': self.topics_discussed,
            'session_topic'   : self.session_topic
        }

## 11. Full rag_answer() Pipeline

The orchestrator that chains every component into a single callable function.

Flow:
  query + session
    → adaptive top-k
    → embedding cache → sentence transformer
    → Qdrant retrieval
    → similarity gate check
    → emotion-aware reranker
    → combined intelligence call (judge + rewriter + scorer)
    → action router (rewrite / broaden / fallback / crisis / answer)
    → context builder → therapist LLM
    → citation builder
    → session memory update
    → final response


In [34]:
# Maximum retry attempts for rewrite/broaden actions
MAX_RETRIES = 2

# Graceful degradation responses (no API call needed)
FALLBACK_RESPONSES = {
    'crisis_generic' : (
        "I hear you and I want you to know that what you are feeling matters deeply. "
        "You are not alone in this. Please reach out to a crisis support line right now "
        "there are people ready to listen and help. {crisis_number}"
    ),
    'technical'      : (
        "I am having a technical difficulty at the moment. "
        "Please try again in a moment and I am here for you."
    ),
    'no_context'     : (
        "I want to help but I am having difficulty finding the right information right now. "
        "Could you tell me a little more about what you are going through?"
    )
}


def rag_answer(
    query             : str,
    session           : SessionMemory,
    detected_emotion  : str   = 'neutral',
    emotion_confidence: float = 0.5,
    response_tone     : str   = 'Empathetic, warm, supportive',
    language_code     : str   = 'en'
) -> dict:
    """
    Main RAG pipeline entry point.

    Args:
        query              : User message (already passed through Modules 1-3)
        session            : Active SessionMemory instance
        detected_emotion   : From Module 2 output
        emotion_confidence : From Module 2 output
        response_tone      : From Module 2 emotion_meta response_tone field
        language_code      : From Module 1 output

    Returns:
        dict with: answer, citations, action, language, session_summary, metadata
    """

    t_start  = time.time()
    metadata = {
        'query'      : query,
        'emotion'    : detected_emotion,
        'language'   : language_code,
        'retries'    : 0,
        'action'     : None,
        'gate_score' : None,
        'top_k'      : None
    }

    # Extract name from message if not in account
    session.extract_name_from_text(query)

    # Add user turn to memory
    session.add_turn('user', query)

    # Retrieval loop (handles rewrite and broaden actions)
    current_query  = query
    top_k_override = None
    retrieval      = None
    judgment       = None

    for attempt in range(MAX_RETRIES + 1):

        retrieval = advanced_retrieve(
            current_query,
            detected_emotion   = detected_emotion,
            emotion_confidence = emotion_confidence,
            top_k_override     = top_k_override
        )
        metadata['gate_score'] = retrieval['gate_score']
        metadata['top_k']      = retrieval['top_k']

        judgment = combined_intelligence_call(
            query            = current_query,
            chunks           = retrieval['chunks'],
            detected_emotion = detected_emotion,
            gate_passed      = retrieval['gate_passed']
        )
        metadata['action']   = judgment['action']
        metadata['retries']  = attempt


        # Route based on action
        if judgment['action'] == 'crisis':
            break   # skip retries and go straight to crisis response


        elif judgment['action'] == 'answer' or judgment['action'] == 'fallback':
            break   


        elif judgment['action'] == 'rewrite' and attempt < MAX_RETRIES:
            rewritten = judgment.get('rewritten_query')

            if rewritten and rewritten != current_query:
                current_query = rewritten
                continue

            else:
                # No rewritten query provided 
                judgment['action'] = 'fallback'
                break



        elif judgment['action'] == 'broaden' and attempt < MAX_RETRIES:
            top_k_override = (retrieval['top_k'] or 5) + 2
            continue

        else:
            # Max retries reached or unknown action
            judgment['action'] = 'fallback'
            break


    # Crisis response (highest priority)
    crisis_active = (
        judgment['action'] == 'crisis'
        or session.crisis_flag
        or (detected_emotion in ['sadness', 'fear'] and emotion_confidence > 0.95)
    )



    if judgment['action'] == 'crisis' and not session.crisis_flag:
        # First crisis detection in this session
        hotline      = get_crisis_number(session.country_code)
        crisis_msg   = FALLBACK_RESPONSES['crisis_generic'].format(
            crisis_number = f"{hotline['name']}: {hotline['number']}"
        )

        # Still go through therapist LLM for a warm personalized response
        # but pass crisis_active=True so it includes the number naturally

    # Therapist LLM 
    result = therapist_llm(
        query                = current_query,
        chunks               = retrieval['chunks'],
        relevant_indices     = judgment.get('relevant_chunks', list(range(min(3, len(retrieval['chunks']))))),
        judgment_action      = judgment['action'],
        detected_emotion     = detected_emotion,
        emotion_conf         = emotion_confidence,
        response_tone        = response_tone,
        language_code        = language_code,
        user_name            = session.user_name,
        country_code         = session.country_code,
        crisis_active        = crisis_active,
        prior_crisis         = session.crisis_flag,
        conversation_history = session.get_history_for_prompt()
    )




    #  Session memory update 
    retrieved_topics = []
    for c in retrieval['chunks'][:3]:
        retrieved_topics.extend(c.get('topics', []))

    session.update_from_pipeline(
        emotion          = detected_emotion,
        crisis_detected  = crisis_active,
        topics           = list(set(retrieved_topics))
    )
    session.add_turn('assistant', result['answer'])

    metadata['latency_ms'] = round((time.time() - t_start) * 1000)

    return {
        'answer'          : result['answer'],
        'citations'       : result['citations'],
        'action'          : judgment['action'],
        'language'        : language_code,
        'crisis_active'   : crisis_active,
        'session_summary' : session.summary(),
        'metadata'        : metadata
    }

In [36]:
"""## 12. Multi-Turn Conversation Test

Scripted 8-turn conversation that exercises every code path in the pipeline:

Turn 1 : Simple anxiety — standard RAG path
Turn 2 : Follow-up — tests memory and sliding window
Turn 3 : Complex compound question — tests adaptive top-k
Turn 4 : Vague query — tests rewrite action
Turn 5 : Name introduction — tests name extraction
Turn 6 : Arabic message — tests language awareness
Turn 7 : Escalating distress — tests crisis flag activation
Turn 8 : Seemingly normal question after crisis — tests sticky crisis flag
"""

# Simulate Module 2 outputs for each turn

SIMULATED_MODULE2 = [
    {'emotion': 'fear',    'confidence': 0.88, 'tone': 'Reassuring, grounding, structured'},
    {'emotion': 'fear',    'confidence': 0.82, 'tone': 'Reassuring, grounding, structured'},
    {'emotion': 'sadness', 'confidence': 0.79, 'tone': 'Warm, validating, gentle'},
    {'emotion': 'sadness', 'confidence': 0.91, 'tone': 'Warm, validating, gentle'},
    {'emotion': 'joy',     'confidence': 0.65, 'tone': 'Encouraging, celebratory, engaging'},
    {'emotion': 'fear',    'confidence': 0.87, 'tone': 'Reassuring, grounding, structured'},
    {'emotion': 'sadness', 'confidence': 0.99, 'tone': 'Warm, validating, gentle'},
    {'emotion': 'sadness', 'confidence': 0.72, 'tone': 'Warm, validating, gentle'},
]

CONVERSATION = [
    {'turn': 1, 'query': 'I have been feeling very anxious lately and I cannot sleep',                   'lang': 'en'},
    {'turn': 2, 'query': 'Can you tell me more about the breathing techniques you mentioned',            'lang': 'en'},
    {'turn': 3, 'query': 'Why do I feel so anxious at work and how does CBT help with that specifically','lang': 'en'},
    {'turn': 4, 'query': 'things are just really bad right now',                                         'lang': 'en'},
    {'turn': 5, 'query': 'By the way my name is Ahmed and I really appreciate your help',                'lang': 'en'},
    {'turn': 6, 'query': 'اشعر بالقلق الشديد ولا اعرف ماذا افعل',                                      'lang': 'ar'},
    {'turn': 7, 'query': 'I feel like there is no point anymore and I want everything to stop',          'lang': 'en'},
    {'turn': 8, 'query': 'Can you recommend some good sleep habits',                                     'lang': 'en'},
]

In [37]:
# Initialize session (simulating a user account: name = None until detected and  country = EG)
session = SessionMemory(user_name = None, country_code = 'EG')

print('Multi-Turn Conversation Test')
print(f'Session ID : {session.session_id}')
print('=' * 80)

for turn_data, m2 in zip(CONVERSATION, SIMULATED_MODULE2):
    print(f"\nTurn {turn_data['turn']}")
    print(f"  User     : {turn_data['query']}")
    print()

    response = rag_answer(
        query              = turn_data['query'],
        session            = session,
        detected_emotion   = m2['emotion'],
        emotion_confidence = m2['confidence'],
        response_tone      = m2['tone'],
        language_code      = turn_data['lang']
    )

    print(f"  Action   : {response['action']}")
    print(f"  Crisis   : {response['crisis_active']}")
    print(f"  Language : {response['language']}")
    print(f"  Latency  : {response['metadata']['latency_ms']} ms")
    print(f"  Retries  : {response['metadata']['retries']}")
    if session.user_name:
        print(f"  Name     : {session.user_name}  (detected: Turn {turn_data['turn']})" if turn_data['turn'] <= 5 else f"  Name     : {session.user_name}")
    print()
    print(f"  Assistant:")
    for line in response['answer'].split('\n'):
        if line.strip():
            print(f"    {line}")
    if response['citations']:
        print(f"\n  Citations : {len(response['citations'])} chunks")
        for c in response['citations']:
            print(f"    topics={c['topics']}  risk={c['risk_level']}  score={c['score']}")
    print()
    print('-' * 80)

print()
print('Session Summary:')
summary = session.summary()
for k, v in summary.items():
    print(f'  {k:<20} : {v}')

Multi-Turn Conversation Test
Session ID : 1a1b6753

Turn 1
  User     : I have been feeling very anxious lately and I cannot sleep

  Action   : answer
  Crisis   : False
  Language : en
  Latency  : 1993 ms
  Retries  : 0

  Assistant:
    I can sense the fear and anxiety that's been weighing on you lately, and I want you to know that I'm here to listen and support you. It takes a lot of courage to acknowledge and share your struggles, so thank you for reaching out. Research in counseling suggests that anxiety can often manifest physically, such as through sleep disturbances, and it's not uncommon for people to experience a sense of overwhelm and irritability when they're feeling anxious.
    It sounds like you're feeling really overwhelmed right now, and it's affecting your ability to sleep. I want to reassure you that you're not alone in this feeling. Many people experience anxiety and sleep disturbances, and there are ways to address these issues. A counselor or therapist can help 

## 13. Save All Artifacts

Saves everything the deployment layer needs to load and run the pipeline
without re-running any notebook cells

In [38]:

ARTIFACTS_DIR = MODULE_4_DIR / 'rag_artifacts'
ARTIFACTS_DIR.mkdir(exist_ok = True)

# RAG configuration  (all thresholds and model names in one place)
rag_config = {
    'embed_model'       : EMBED_MODEL_NAME,
    'vector_dim'        : VECTOR_DIM,
    'collection_name'   : COLLECTION_NAME,
    'similarity_gate'   : SIMILARITY_GATE,
    'judge_model'       : JUDGE_MODEL,
    'answer_model'      : ANSWER_MODEL,
    'max_retries'       : MAX_RETRIES,
    'window_size'       : SessionMemory.WINDOW_SIZE,
    'emotion_weights'   : EMOTION_RERANK_WEIGHTS
}

with open(ARTIFACTS_DIR / 'rag_config.json', 'w') as f:
    json.dump(rag_config, f, indent = 2)
print(f'RAG config saved           : {ARTIFACTS_DIR}/rag_config.json')



# Therapist system prompt (for auditability and version control)
with open(ARTIFACTS_DIR / 'therapist_system_prompt.txt', 'w', encoding = 'utf-8') as f:
    f.write(THERAPIST_SYSTEM_PROMPT)
print(f'Therapist prompt saved     : {ARTIFACTS_DIR}/therapist_system_prompt.txt')



# Judge system prompt
with open(ARTIFACTS_DIR / 'judge_system_prompt.txt', 'w', encoding = 'utf-8') as f:
    f.write(JUDGE_SYSTEM_PROMPT)
print(f'Judge prompt saved         : {ARTIFACTS_DIR}/judge_system_prompt.txt')



# Crisis hotlines
with open(ARTIFACTS_DIR / 'crisis_hotlines.json', 'w', encoding = 'utf-8') as f:
    json.dump(CRISIS_HOTLINES, f, indent = 2, ensure_ascii = False)
print(f'Crisis hotlines saved      : {ARTIFACTS_DIR}/crisis_hotlines.json')



# Fallback responses
with open(ARTIFACTS_DIR / 'fallback_responses.json', 'w', encoding = 'utf-8') as f:
    json.dump(FALLBACK_RESPONSES, f, indent = 2, ensure_ascii = False)
print(f'Fallback responses saved   : {ARTIFACTS_DIR}/fallback_responses.json')



# Session store directory (deployment uses this to persist sessions)
session_store = ARTIFACTS_DIR / 'session_store'
session_store.mkdir(exist_ok = True)
print(f'Session store created      : {session_store}')



print()
print('All artifacts saved.')
print()
print(f'Artifact directory contents:')
for f in sorted(ARTIFACTS_DIR.iterdir()):
    size = f.stat().st_size / 1024
    print(f'  {f.name:<40} {size:.1f} KB')

RAG config saved           : ..\module_4_rag\rag_artifacts/rag_config.json
Therapist prompt saved     : ..\module_4_rag\rag_artifacts/therapist_system_prompt.txt
Judge prompt saved         : ..\module_4_rag\rag_artifacts/judge_system_prompt.txt
Crisis hotlines saved      : ..\module_4_rag\rag_artifacts/crisis_hotlines.json
Fallback responses saved   : ..\module_4_rag\rag_artifacts/fallback_responses.json
Session store created      : ..\module_4_rag\rag_artifacts\session_store

All artifacts saved.

Artifact directory contents:
  crisis_hotlines.json                     5.5 KB
  fallback_responses.json                  0.5 KB
  judge_system_prompt.txt                  1.5 KB
  rag_config.json                          1.1 KB
  session_store                            0.0 KB
  therapist_system_prompt.txt              1.9 KB
